# AI Assignment — General-Purpose Classical Reasoning Agent

**Pipeline:** Knowledge base (FOL) → Parser → AST → Clausal form (NNF, Skolemisation, CNF) → **Classical reasoning** → `True` / `False`

The evaluation supplies the knowledge base and the query **directly as first-order logic**, so the system is built around a robust FOL parser and a self-implemented reasoning engine. No generative model is used anywhere; the final decision always comes from the engine below.

| Component | Technique |
|---|---|
| Parsing | Recursive-descent parser for unicode / ASCII / Prolog FOL syntax |
| Normal form | NNF → standardising apart → Skolemisation → CNF (Tseitin renaming against blow-up) |
| Matching | Robinson **unification** with occurs check |
| Horn KBs (facts + if-then rules) | Semi-naive **forward chaining** to the least Herbrand model, with proof trees |
| General function-free FOL (∨, ¬, ⊕, ↔, ∃ …) | **Herbrand grounding + DPLL** (DFS over assignments, unit propagation with two watched literals). This is a complete decision procedure here. |
| FOL with function symbols | Depth-bounded Herbrand grounding (iterative deepening), then **resolution refutation** (given-clause, set of support, best-first by clause weight, factoring) |
| Closed-world data (optional) | Stratified negation-as-failure forward chaining |

**Decision rule (binary task):** `True` iff KB ⊨ query, i.e. KB ∧ ¬query is unsatisfiable. If the KB entails ¬query, or neither, the answer is `False`.

**To evaluate on the private test set:** set `TEST_PATH` in the last section and run all cells. The notebook prints the final accuracy and inference time and writes `submission.csv` (`id, flag`).

## 0. Configuration

In [1]:
import re, ast, json, time, itertools, heapq, os, sys
from collections import defaultdict, deque

CONFIG = {
    # Classical (open-world) entailment: True iff KB ⊨ query, otherwise False
    # (covers both "KB ⊨ ¬query" and "unknown").  Set True only for
    # closed-world data where "not X" means "X is not derivable".
    'closed_world': False,
    # treat 'Rina' and 'rina' as the same constant / predicate
    'normalize_case': True,
    # translate plain-English contexts with deterministic templates
    # (only used if the input is NOT already logic)
    'nl_fallback': True,
    'max_ground_clauses': 300_000,   # grounding budget before switching to resolution
    'herbrand_max_depth': 1,         # nesting depth of function terms tried by grounding
    'resolution_max_steps': 1500,
    'resolution_time_limit': 0.5,    # seconds per query (resolution fallback only)
    'default_answer': False,         # answer if an input cannot be parsed at all
}
sys.setrecursionlimit(10000)

## 1. FOL parser
Accepts `∀x (P(x) → Q(x))`, `forall x. p(x) -> q(x)`, `q(X) :- p(X).`, `⊕/xor`, `↔/<->`, `=`/`≠`, functions, 0-ary propositions, lists of premises. Free `x, y, z…` are read as universally quantified. Unbalanced brackets are repaired.

In [2]:
# ============================================================
# 1. FOL PARSER  (text  ->  abstract syntax tree)
# ------------------------------------------------------------
# Accepts the common notations a logic KB may be written in:
#   unicode : ∀x (Young(x) → Smart(x))      ∃x P(x)   ¬  ∧  ∨  ⊕  ↔
#   ascii   : forall x. young(x) -> smart(x)   exists x   ~ ! not  & and  | or
#             => implies   <-> <=> iff   xor   = != ≠
#   prolog  : smart(X) :- young(X), kind(X).
# AST nodes (tuples):
#   ('atom', pred, (terms...))     ('not', f)   ('and', f, g)  ('or', f, g)
#   ('imp', f, g)  ('iff', f, g)   ('xor', f, g)
#   ('all', var, f) ('ex', var, f)
# Terms: ('var', name) | ('const', name) | ('fn', name, (terms...))
# ============================================================

_TOKEN_SPEC = [
    ('WS',     r'\s+'),
    ('IFF',    r'<->|<=>|↔|⇔|≡|\biff\b|\bIFF\b'),
    ('RIMP',   r':-|←'),
    ('IMP',    r'->|=>|→|⇒|⊃|⟹|\bimplies\b|\bIMPLIES\b'),
    ('XOR',    r'⊕|⊻|<\+>|\bxor\b|\bXOR\b'),
    ('AND',    r'∧|&&|&|/\\|\^|\band\b|\bAND\b'),
    ('OR',     r'∨|\|\||\||\\/|\bor\b|\bOR\b'),
    ('NEQ',    r'≠|!=|\\=|/='),
    ('NOT',    r'¬|~|!|\\\+|\bnot\b|\bNOT\b|-(?!>)'),
    ('EQ',     r'==|='),
    ('FORALL', r'∀|\\forall\b|\bforall\b|\bFORALL\b|\bForall\b'),
    ('EXISTS', r'∃|\\exists\b|\bexists\b|\bEXISTS\b|\bExists\b'),
    ('LP',     r'[\(\[\{]'),
    ('RP',     r'[\)\]\}]'),
    ('COMMA',  r','),
    ('DOT',    r'\.'),
    ('COLON',  r':'),
    ('ID',     r"\d+\.\d+|[\w][\w']*(?:-(?!>)[\w']+)*"),
    ('MISC',   r'.'),
]
_TOKEN_RE = re.compile('|'.join('(?P<%s>%s)' % p for p in _TOKEN_SPEC), re.UNICODE)

# identifiers that are treated as implicitly universally quantified
# variables when they appear free (unbound by any quantifier)
_FREE_VAR_RE = re.compile(r'^(?:[xyzuvw]\d*|[XYZUVW]\d*|_\w*)$')


class FOLParseError(Exception):
    pass


def tokenize(text):
    toks = []
    for m in _TOKEN_RE.finditer(text):
        kind = m.lastgroup
        if kind == 'WS':
            continue
        if kind == 'MISC':
            ch = m.group()
            if ch in '"`’‘“”':          # stray quote characters -> ignore
                continue
            raise FOLParseError('unexpected character %r' % ch)
        toks.append((kind, m.group()))
    return toks


class _Parser:
    def __init__(self, tokens, prolog_vars=False):
        self.t = tokens
        self.i = 0
        self.scope = []                # stack of bound variable names
        self.prolog_vars = prolog_vars  # uppercase identifiers are variables

    # ---- helpers -------------------------------------------------
    def peek(self, k=0):
        j = self.i + k
        return self.t[j][0] if j < len(self.t) else None

    def val(self, k=0):
        j = self.i + k
        return self.t[j][1] if j < len(self.t) else None

    def eat(self, kind=None):
        if self.i >= len(self.t):
            raise FOLParseError('unexpected end of formula')
        tok = self.t[self.i]
        if kind and tok[0] != kind:
            raise FOLParseError('expected %s got %r' % (kind, tok[1]))
        self.i += 1
        return tok

    # ---- grammar (lowest -> highest precedence) -----------------
    #  iff/xor  <  imp (right assoc)  <  or  <  and  <  unary
    def formula(self):
        left = self.imp()
        while self.peek() in ('IFF', 'XOR'):
            op = 'iff' if self.eat()[0] == 'IFF' else 'xor'
            left = (op, left, self.imp())
        return left

    def imp(self):
        left = self.disj()
        if self.peek() == 'IMP':
            self.eat()
            return ('imp', left, self.imp())
        if self.peek() == 'RIMP':           # head :- body   /  head ← body
            self.eat()
            return ('imp', self.imp(), left)
        return left

    def disj(self):
        left = self.conj()
        while self.peek() == 'OR':
            self.eat()
            left = ('or', left, self.conj())
        return left

    def conj(self):
        left = self.unary()
        while self.peek() == 'AND':
            self.eat()
            left = ('and', left, self.unary())
        return left

    def unary(self):
        k = self.peek()
        if k == 'NOT':
            self.eat()
            return ('not', self.unary())
        if k in ('FORALL', 'EXISTS'):
            q = 'all' if self.eat()[0] == 'FORALL' else 'ex'
            names = [self.eat('ID')[1]]
            while self.peek() == 'COMMA' and self.peek(1) == 'ID':
                self.eat()
                names.append(self.eat('ID')[1])
            while self.peek() in ('DOT', 'COLON'):
                self.eat()
            self.scope.extend(names)
            # quantifier scope extends as far to the right as possible
            body = self.formula()
            del self.scope[-len(names):]
            for n in reversed(names):
                body = (q, n, body)
            return body
        if k == 'LP':
            self.eat()
            f = self.formula()
            self.eat('RP')
            return f
        return self.atom()

    def atom(self):
        t1 = self.term()
        if self.peek() in ('EQ', 'NEQ'):
            neg = self.eat()[0] == 'NEQ'
            t2 = self.term()
            a = ('atom', '=', (t1, t2))
            return ('not', a) if neg else a
        if t1[0] == 'fn':
            return ('atom', t1[1], t1[2])
        if t1[0] == 'const':              # 0-ary predicate / proposition
            return ('atom', t1[1], ())
        raise FOLParseError('variable %r used as a formula' % (t1[1],))

    def term(self):
        name = self.eat('ID')[1]
        if self.peek() == 'LP':
            self.eat()
            args = []
            if self.peek() != 'RP':
                args.append(self.term())
                while self.peek() == 'COMMA':
                    self.eat()
                    args.append(self.term())
            self.eat('RP')
            return ('fn', name, tuple(args))
        if name in self.scope:
            return ('var', name)
        if self.prolog_vars and (name[0].isupper() or name[0] == '_'):
            return ('var', name)
        return ('const', name)


# ---------- free-variable closure ---------------------------------
def _free_var_candidates(f, acc, bound=frozenset()):
    t = f[0]
    if t == 'atom':
        for a in f[2]:
            _term_consts(a, acc, bound)
    elif t == 'not':
        _free_var_candidates(f[1], acc, bound)
    elif t in ('all', 'ex'):
        _free_var_candidates(f[2], acc, bound | {f[1]})
    else:
        _free_var_candidates(f[1], acc, bound)
        _free_var_candidates(f[2], acc, bound)


def _term_consts(t, acc, bound):
    if t[0] == 'const':
        acc.add(t[1])
    elif t[0] == 'var':
        if t[1] not in bound:
            acc.add(('VAR', t[1]))
    elif t[0] == 'fn':
        for a in t[2]:
            _term_consts(a, acc, bound)


def _const_to_var(f, names):
    t = f[0]
    if t == 'atom':
        return ('atom', f[1], tuple(_t_c2v(a, names) for a in f[2]))
    if t == 'not':
        return ('not', _const_to_var(f[1], names))
    if t in ('all', 'ex'):
        return (t, f[1], _const_to_var(f[2], names - {f[1]}))
    return (t, _const_to_var(f[1], names), _const_to_var(f[2], names))


def _t_c2v(t, names):
    if t[0] == 'const' and t[1] in names:
        return ('var', t[1])
    if t[0] == 'fn':
        return ('fn', t[1], tuple(_t_c2v(a, names) for a in t[2]))
    return t


def _close_free_vars(f):
    """x, y, z ... (and X, Y, Z) that are not bound by any quantifier are
    read as universally quantified variables (standard convention)."""
    acc = set()
    _free_var_candidates(f, acc)
    free = sorted(n for n in acc if isinstance(n, str) and _FREE_VAR_RE.match(n))
    # variables already marked (prolog style) but unbound
    free_v = sorted(n[1] for n in acc if isinstance(n, tuple))
    if not free and not free_v:
        return f
    f = _const_to_var(f, set(free))
    for n in reversed(sorted(set(free) | set(free_v))):
        f = ('all', n, f)
    return f


def _normalize_case(f):
    """Lower-case predicate and constant names so that 'Rina' and 'rina'
    (a frequent annotation inconsistency) denote the same symbol."""
    t = f[0]
    if t == 'atom':
        p = f[1] if f[1] == '=' else f[1].lower()
        return ('atom', p, tuple(_nc_term(a) for a in f[2]))
    if t == 'not':
        return ('not', _normalize_case(f[1]))
    if t in ('all', 'ex'):
        return (t, f[1], _normalize_case(f[2]))
    return (t, _normalize_case(f[1]), _normalize_case(f[2]))


def _nc_term(t):
    if t[0] == 'const':
        return ('const', t[1].lower())
    if t[0] == 'fn':
        return ('fn', t[1].lower(), tuple(_nc_term(a) for a in t[2]))
    return t


def _prolog_rewrite(s):
    """head :- b1, b2.   ->   (b1 ∧ b2) → head   (commas at depth 0 = ∧)."""
    if ':-' not in s:
        return s, False
    head, body = s.split(':-', 1)
    parts, depth, cur = [], 0, ''
    for ch in body:
        if ch in '([{':
            depth += 1
        elif ch in ')]}':
            depth -= 1
        if ch == ',' and depth == 0:
            parts.append(cur)
            cur = ''
        else:
            cur += ch
    parts.append(cur)
    body = ' ∧ '.join('(%s)' % p.strip() for p in parts if p.strip())
    return '(%s) → (%s)' % (body, head.strip()), True


def parse_formula(text):
    """Parse ONE formula.  Tolerates trailing '.', unbalanced brackets."""
    s = text.strip()
    s = s.rstrip('.').strip()
    if not s:
        raise FOLParseError('empty formula')
    s, prolog = _prolog_rewrite(s)
    attempts = [s]
    opened, closed = s.count('(') + s.count('['), s.count(')') + s.count(']')
    if opened > closed:
        attempts.append(s + ')' * (opened - closed))
    elif closed > opened:
        attempts.append('(' * (closed - opened) + s)
        t = s
        for _ in range(closed - opened):
            if t.endswith(')'):
                t = t[:-1]
        attempts.append(t)
    last_err = None
    for a in attempts:
        try:
            toks = tokenize(a)
            p = _Parser(toks, prolog_vars=prolog)
            f = p.formula()
            while p.peek() == 'DOT':
                p.eat()
            if p.i != len(toks):
                raise FOLParseError('trailing tokens: %s' %
                                    ' '.join(v for _, v in toks[p.i:]))
            f = _close_free_vars(f)
            if CONFIG['normalize_case']:
                f = _normalize_case(f)
            return f
        except FOLParseError as e:
            last_err = e
    raise FOLParseError('%s  <<%s>>' % (last_err, text))


_QUANT_TAIL = re.compile(r'(∀|∃|\bforall|\bexists)\s*\w+(\s*,\s*\w+)*\s*$', re.I)


def split_premises(ctx):
    """A context may be a python list, a JSON list string, or one string with
    formulas separated by newlines, ';' or sentence-final '.'."""
    if ctx is None:
        return []
    if isinstance(ctx, (list, tuple)):
        out = []
        for c in ctx:
            out.extend(split_premises(c))
        return out
    if hasattr(ctx, 'tolist'):                      # numpy array from pandas
        return split_premises(list(ctx.tolist()))
    s = str(ctx).strip()
    if s.startswith('[') and s.endswith(']') and ('"' in s or "'" in s):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return split_premises(list(v))
        except Exception:
            try:
                v = json.loads(s)
                if isinstance(v, list):
                    return split_premises(v)
            except Exception:
                pass
    pieces = []
    for line in re.split(r'[\n;]+', s):
        # split on '.' at bracket depth 0 that ends a formula/sentence
        depth, cur = 0, ''
        for i, ch in enumerate(line):
            if ch in '([{':
                depth += 1
            elif ch in ')]}':
                depth -= 1
            if ch == '.' and depth <= 0:
                nxt = line[i + 1] if i + 1 < len(line) else ' '
                if nxt.isspace() and not _QUANT_TAIL.search(cur) \
                        and not re.search(r'\d$', cur):
                    if cur.strip():
                        pieces.append(cur.strip())
                    cur = ''
                    continue
            cur += ch
        if cur.strip():
            pieces.append(cur.strip())
    return [p for p in pieces if p.strip(' .')]


def fol_to_str(f):
    """Pretty printer (used in traces)."""
    t = f[0]
    if t == 'atom':
        if f[1] == '=':
            return '%s = %s' % (term_str(f[2][0]), term_str(f[2][1]))
        return f[1] if not f[2] else '%s(%s)' % (f[1], ', '.join(term_str(a) for a in f[2]))
    if t == 'not':
        return '¬' + fol_to_str(f[1])
    sym = {'and': '∧', 'or': '∨', 'imp': '→', 'iff': '↔', 'xor': '⊕'}
    if t in sym:
        return '(%s %s %s)' % (fol_to_str(f[1]), sym[t], fol_to_str(f[2]))
    return '%s%s %s' % ('∀' if t == 'all' else '∃', f[1], fol_to_str(f[2]))


def term_str(t):
    if isinstance(t, tuple) and t and t[0] in ('var', 'const'):
        return t[1]
    if isinstance(t, tuple) and t and t[0] == 'fn':
        return '%s(%s)' % (t[1], ', '.join(term_str(a) for a in t[2]))
    # internal clause-level term representation
    if isinstance(t, str):
        return t
    return '%s(%s)' % (t[0], ', '.join(term_str(a) for a in t[1:]))

## 2. Clausal normal form (NNF → Skolemisation → CNF)

In [3]:
# ============================================================
# 2. CLAUSAL NORMAL FORM
#    NNF  ->  standardise variables apart  ->  Skolemise  ->  CNF
# ------------------------------------------------------------
# Clause-level representation (compact & hashable, fast to unify):
#   variable  : str starting with '?'          e.g. '?v3'
#   constant  : plain str                      e.g. 'bob'
#   function  : tuple (fname, arg1, ...)       e.g. ('$sk2', '?v1')
#   literal   : (sign: bool, pred: str, args: tuple)
#   clause    : frozenset of literals
# ============================================================

class _Fresh:
    def __init__(self):
        self.n = 0

    def __call__(self, prefix):
        self.n += 1
        return '%s%d' % (prefix, self.n)


_fresh = _Fresh()


def to_nnf(f, pos=True):
    t = f[0]
    if t == 'atom':
        return f if pos else ('not', f)
    if t == 'not':
        return to_nnf(f[1], not pos)
    if t == 'and':
        a, b = to_nnf(f[1], pos), to_nnf(f[2], pos)
        return ('and', a, b) if pos else ('or', a, b)
    if t == 'or':
        a, b = to_nnf(f[1], pos), to_nnf(f[2], pos)
        return ('or', a, b) if pos else ('and', a, b)
    if t == 'imp':                       # a → b   ≡   ¬a ∨ b
        return to_nnf(('or', ('not', f[1]), f[2]), pos)
    if t in ('iff', 'xor'):
        a, b = f[1], f[2]
        if (t == 'iff') == pos:          # a ↔ b ≡ (¬a ∨ b) ∧ (a ∨ ¬b)
            return ('and', ('or', to_nnf(a, False), to_nnf(b, True)),
                           ('or', to_nnf(a, True), to_nnf(b, False)))
        else:                            # a ⊕ b ≡ (a ∨ b) ∧ (¬a ∨ ¬b)
            return ('and', ('or', to_nnf(a, True), to_nnf(b, True)),
                           ('or', to_nnf(a, False), to_nnf(b, False)))
    if t == 'all':
        return ('all' if pos else 'ex', f[1], to_nnf(f[2], pos))
    if t == 'ex':
        return ('ex' if pos else 'all', f[1], to_nnf(f[2], pos))
    raise ValueError(t)


def _ast_term(t, env):
    """AST term -> clause-level term under variable environment env."""
    k = t[0]
    if k == 'var':
        return env.get(t[1], t[1])
    if k == 'const':
        return t[1]
    return (t[1],) + tuple(_ast_term(a, env) for a in t[2])


def _free_names(f, bound=frozenset()):
    t = f[0]
    if t == 'atom':
        s = set()
        for a in f[2]:
            _vars_in_ast_term(a, s)
        return s - bound
    if t == 'not':
        return _free_names(f[1], bound)
    if t in ('all', 'ex'):
        return _free_names(f[2], bound | {f[1]})
    return _free_names(f[1], bound) | _free_names(f[2], bound)


def _vars_in_ast_term(t, s):
    if t[0] == 'var':
        s.add(t[1])
    elif t[0] == 'fn':
        for a in t[2]:
            _vars_in_ast_term(a, s)


def skolemize(f, env=None, univ=()):
    """f must be in NNF.  Returns a quantifier-free NNF formula over
    clause-level terms.  Skolem functions only take the universal variables
    that actually occur free in the scope of the ∃ (smaller Herbrand
    universe => grounding stays function-free much more often)."""
    env = env or {}
    t = f[0]
    if t == 'atom':
        return ('atom', f[1], tuple(_ast_term(a, env) for a in f[2]))
    if t == 'not':
        return ('not', skolemize(f[1], env, univ))
    if t in ('and', 'or'):
        return (t, skolemize(f[1], env, univ), skolemize(f[2], env, univ))
    if t == 'all':
        v = _fresh('?v')
        e2 = dict(env)
        e2[f[1]] = v
        return skolemize(f[2], e2, univ + ((f[1], v),))
    if t == 'ex':
        free = _free_names(f[2]) - {f[1]}
        deps = tuple(v for (name, v) in univ if name in free and env.get(name) == v)
        sk = _fresh('$sk')
        e2 = dict(env)
        e2[f[1]] = sk if not deps else (sk,) + deps
        return skolemize(f[2], e2, univ)
    raise ValueError(t)


def _lit(f):
    if f[0] == 'not':
        return (False, f[1][1], f[1][2])
    return (True, f[1], f[2])


def _is_taut(cl):
    for (s, p, a) in cl:
        if s and (False, p, a) in cl:
            return True
    return False


_DISTRIB_LIMIT = 256


def _cnf(f, out_defs):
    """Return list of clauses (frozensets).  Uses Tseitin-style renaming when
    distribution would blow up (sound for refutation: equisatisfiable)."""
    t = f[0]
    if t in ('atom', 'not'):
        return [frozenset([_lit(f)])]
    if t == 'and':
        return _cnf(f[1], out_defs) + _cnf(f[2], out_defs)
    if t == 'or':
        A, B = _cnf(f[1], out_defs), _cnf(f[2], out_defs)
        if len(A) * len(B) > _DISTRIB_LIMIT:
            # name the bigger side:  N(vars) → side
            if len(A) < len(B):
                A, B = B, A
            vs = sorted(set().union(*[clause_vars(c) for c in A]))
            name = _fresh('$def')
            n_lit = (True, name, tuple(vs))
            for c in A:
                out_defs.append(c | {(False, name, tuple(vs))})
            A = [frozenset([n_lit])]
        res = []
        for a in A:
            for b in B:
                c = a | b
                if not _is_taut(c):
                    res.append(c)
        return res
    raise ValueError(t)


def formula_to_clauses(f, negate=False):
    g = to_nnf(f, pos=not negate)
    g = skolemize(g)
    defs = []
    cls = _cnf(g, defs)
    return [c for c in cls + defs if not _is_taut(c)]


# ---------- term / clause utilities ---------------------------------
def is_var(t):
    return isinstance(t, str) and t[:1] == '?'


def term_vars(t, acc):
    if isinstance(t, str):
        if t[:1] == '?':
            acc.add(t)
    else:
        for a in t[1:]:
            term_vars(a, acc)


def clause_vars(c):
    acc = set()
    for (_, _, args) in c:
        for a in args:
            term_vars(a, acc)
    return acc


def has_functions(clauses):
    for c in clauses:
        for (_, _, args) in c:
            for a in args:
                if not isinstance(a, str):
                    return True
    return False


def clause_constants(clauses):
    acc = set()

    def walk(t):
        if isinstance(t, str):
            if t[:1] != '?':
                acc.add(t)
        else:
            for a in t[1:]:
                walk(a)
    for c in clauses:
        for (_, _, args) in c:
            for a in args:
                walk(a)
    return acc


def lit_str(l):
    s, p, a = l
    if p == '=' and len(a) == 2:
        core = '%s = %s' % (term_str(a[0]), term_str(a[1]))
    else:
        core = p if not a else '%s(%s)' % (p, ', '.join(term_str(x) for x in a))
    return core if s else '¬' + core


def clause_str(c):
    if not c:
        return '□'
    return ' ∨ '.join(lit_str(l) for l in sorted(c, key=lambda l: (l[1], not l[0], str(l[2]))))

## 3–5. Unification, forward chaining (Horn), closed-world forward chaining

In [4]:
# ============================================================
# 3. UNIFICATION  (Robinson, with occurs check)
# ============================================================

def walk(t, s):
    while isinstance(t, str) and t[:1] == '?' and t in s:
        t = s[t]
    return t


def occurs(v, t, s):
    t = walk(t, s)
    if t == v:
        return True
    if isinstance(t, tuple):
        return any(occurs(v, a, s) for a in t[1:])
    return False


def unify(a, b, s):
    """Returns extended substitution dict or None."""
    stack = [(a, b)]
    s = dict(s)
    while stack:
        x, y = stack.pop()
        x, y = walk(x, s), walk(y, s)
        if x == y:
            continue
        if isinstance(x, str) and x[:1] == '?':
            if occurs(x, y, s):
                return None
            s[x] = y
        elif isinstance(y, str) and y[:1] == '?':
            if occurs(y, x, s):
                return None
            s[y] = x
        elif isinstance(x, tuple) and isinstance(y, tuple) \
                and x[0] == y[0] and len(x) == len(y):
            stack.extend(zip(x[1:], y[1:]))
        else:
            return None
    return s


def unify_args(xs, ys, s):
    if len(xs) != len(ys):
        return None
    return unify(('$', ) + tuple(xs), ('$', ) + tuple(ys), s)


def subst(t, s):
    t = walk(t, s)
    if isinstance(t, tuple):
        return (t[0],) + tuple(subst(a, s) for a in t[1:])
    return t


def subst_lit(l, s):
    return (l[0], l[1], tuple(subst(a, s) for a in l[2]))


# ============================================================
# 4. FORWARD CHAINING  (definite Horn clauses, semi-naive, with proofs)
# ------------------------------------------------------------
# Used as the fast path whenever every clause of KB is a definite clause
# (exactly one positive literal) and the KB is function-free; this is the
# typical "facts + if-then rules" knowledge base.  The least Herbrand model
# is computed once per KB and cached, so many queries over one KB are cheap.
# ============================================================

class HornModel:
    def __init__(self, clauses):
        self.facts = defaultdict(set)      # pred -> set(args)
        self.why = {}                      # (pred,args) -> (rule_idx, [premise atoms])
        self.rules = []                    # (body[(pred,args)], head(pred,args))
        by_pred = defaultdict(list)        # pred -> [(rule_idx, body_pos)]
        agenda = deque()
        for c in clauses:
            pos = [l for l in c if l[0]]
            neg = [l for l in c if not l[0]]
            head = (pos[0][1], pos[0][2])
            if not neg:
                if clause_vars(c):
                    # non-ground fact  ∀x P(x)  -> rule with empty body, grounded later
                    self.rules.append(([], head))
                else:
                    if head[1] not in self.facts[head[0]]:
                        self.facts[head[0]].add(head[1])
                        self.why[head] = ('given', [])
                        agenda.append(head)
            else:
                body = [(l[1], l[2]) for l in neg]
                ri = len(self.rules)
                self.rules.append((body, head))
                for bi, (p, _) in enumerate(body):
                    by_pred[p].append((ri, bi))
        self.consts = clause_constants(clauses) or {'$c0'}
        # 'unsafe' rules quantify over the whole domain (head var not in body)
        self.unsafe = any(set(a for a in h[1] if is_var(a)) -
                          set(a for (_, args) in b for a in args if is_var(a))
                          for b, h in self.rules)
        # body-less non-ground rules: instantiate over the Herbrand universe
        for ri, (body, head) in enumerate(self.rules):
            if not body:
                vs = sorted({a for a in head[1] if is_var(a)})
                for combo in itertools.product(sorted(self.consts), repeat=len(vs)):
                    s = dict(zip(vs, combo))
                    g = (head[0], tuple(subst(a, s) for a in head[1]))
                    if g[1] not in self.facts[g[0]]:
                        self.facts[g[0]].add(g[1])
                        self.why[g] = ('given', [])
                        agenda.append(g)
        self.steps = 0
        # semi-naive: each new fact is joined only with rules that mention its predicate
        while agenda:
            fact = agenda.popleft()
            for ri, bi in by_pred.get(fact[0], ()):
                body, head = self.rules[ri]
                s0 = unify_args(body[bi][1], fact[1], {})
                if s0 is None:
                    continue
                rest = body[:bi] + body[bi + 1:]
                for s in self._join(rest, s0):
                    self.steps += 1
                    hargs = tuple(subst(a, s) for a in head[1])
                    if any(is_var(a) for a in hargs):
                        # head variable not bound by body: range over universe
                        vs = sorted({a for a in hargs if is_var(a)})
                        insts = [tuple(subst(a, dict(zip(vs, cmb))) for a in hargs)
                                 for cmb in itertools.product(sorted(self.consts), repeat=len(vs))]
                    else:
                        insts = [hargs]
                    for ha in insts:
                        if ha not in self.facts[head[0]]:
                            self.facts[head[0]].add(ha)
                            prem = [(p, tuple(subst(a, s) for a in args)) for (p, args) in body]
                            self.why[(head[0], ha)] = (ri, prem)
                            agenda.append((head[0], ha))

    def _join(self, atoms, s):
        if not atoms:
            yield s
            return
        (p, args), rest = atoms[0], atoms[1:]
        bound = tuple(subst(a, s) for a in args)
        if not any(is_var(a) for a in bound):
            if bound in self.facts.get(p, ()):
                yield from self._join(rest, s)
            return
        for fa in list(self.facts.get(p, ())):
            s2 = unify_args(bound, fa, s)
            if s2 is not None:
                yield from self._join(rest, s2)

    def holds(self, atoms):
        """Existential conjunctive query: returns a substitution or None."""
        for s in self._join(list(atoms), {}):
            return s
        return None

    def proof(self, atom, depth=0, seen=None):
        seen = seen if seen is not None else set()
        pad = '  ' * depth
        if atom in seen:
            return []
        seen.add(atom)
        a_str = lit_str((True, atom[0], atom[1]))
        how = self.why.get(atom)
        if how is None:
            return [pad + a_str + '   (?)']
        if how[0] == 'given':
            return [pad + a_str + '   [fact]']
        ri, prem = how
        body, head = self.rules[ri]
        rule_s = ' ∧ '.join(lit_str((True, p, a)) for p, a in body) + ' → ' + \
            lit_str((True, head[0], head[1]))
        lines = [pad + a_str + '   [by rule: %s]' % rule_s]
        for pa in prem:
            lines += self.proof(pa, depth + 1, seen)
        return lines


def is_definite(c):
    return sum(1 for l in c if l[0]) == 1


# ============================================================
# 5. CLOSED-WORLD FORWARD CHAINING  (stratified negation-as-failure)
# ------------------------------------------------------------
# Only used when CONFIG['closed_world'] is True (e.g. the original RuleTaker
# CWA theories where "not big" means "big cannot be derived").
# Rules come straight from the implication structure of each formula.
# ============================================================

def _flatten_and(f):
    if f[0] == 'and':
        return _flatten_and(f[1]) + _flatten_and(f[2])
    return [f]


def _ast_literal(f, env):
    neg = False
    while f[0] == 'not':
        neg = not neg
        f = f[1]
    if f[0] != 'atom':
        return None
    return (not neg, f[1], tuple(_ast_term(a, env) for a in f[2]))


def formula_to_nafrules(f):
    """∀x̄ (L1 ∧ ... ∧ Ln → H1 ∧ ... ∧ Hm)   or  conjunction of literals.
    Returns list of (body_literals, head_literal) or None if not of this shape."""
    env = {}
    while f[0] == 'all':
        env[f[1]] = '?' + f[1]
        f = f[2]
    if f[0] == 'imp':
        body = [_ast_literal(g, env) for g in _flatten_and(f[1])]
        heads = [_ast_literal(g, env) for g in _flatten_and(f[2])]
        if None in body or None in heads:
            return None
        return [(body, h) for h in heads]
    lits = [_ast_literal(g, env) for g in _flatten_and(f)]
    if None in lits:
        return None
    return [([], h) for h in lits]


class CWAModel:
    def __init__(self, rules):
        self.facts = defaultdict(set)
        consts = set()
        for body, head in rules:
            for l in body + [head]:
                for a in l[2]:
                    if not is_var(a):
                        consts.add(a)
        self.consts = sorted(consts) or ['$c0']
        pos_rules = [(b, h) for b, h in rules if h[0]]     # negative heads ignored under CWA
        # stratification over predicates
        preds = {h[1] for _, h in pos_rules} | {l[1] for b, _ in pos_rules for l in b}
        strat = {p: 0 for p in preds}
        for _ in range(len(preds) + 1):
            changed = False
            for b, h in pos_rules:
                for l in b:
                    need = strat[l[1]] + (0 if l[0] else 1)
                    if strat[h[1]] < need:
                        strat[h[1]] = need
                        changed = True
            if not changed:
                break
        levels = sorted(set(strat[h[1]] for _, h in pos_rules)) if pos_rules else []
        for lv in levels:
            rs = [(b, h) for b, h in pos_rules if strat[h[1]] == lv]
            changed = True
            while changed:
                changed = False
                for b, h in rs:
                    for s in self._match(b, {}):
                        args = tuple(subst(a, s) for a in h[2])
                        if args not in self.facts[h[1]]:
                            self.facts[h[1]].add(args)
                            changed = True

    def _match(self, body, s):
        posl = [l for l in body if l[0]]
        negl = [l for l in body if not l[0]]
        for s1 in self._join(posl, s):
            # bind remaining variables over the domain (needed for e.g.
            # "if someone is not big then they are kind")
            rest = set()
            for l in negl:
                for a in l[2]:
                    if is_var(walk(a, s1)):
                        rest.add(walk(a, s1))
            rest = sorted(rest)
            for combo in itertools.product(self.consts, repeat=len(rest)):
                s2 = dict(s1)
                s2.update(zip(rest, combo))
                if all(tuple(subst(a, s2) for a in l[2]) not in self.facts[l[1]] for l in negl):
                    yield s2

    def _join(self, lits, s):
        if not lits:
            yield s
            return
        l, rest = lits[0], lits[1:]
        for fa in list(self.facts.get(l[1], ())):
            s2 = unify_args(l[2], fa, s)
            if s2 is not None:
                yield from self._join(rest, s2)

    def query(self, lits):
        """conjunction of ground/existential literals under CWA"""
        posl = [l for l in lits if l[0]]
        negl = [l for l in lits if not l[0]]
        for s in self._join(posl, {}):
            vs = sorted({walk(a, s) for l in negl for a in l[2] if is_var(walk(a, s))})
            for combo in itertools.product(self.consts, repeat=len(vs)):
                s2 = dict(s)
                s2.update(zip(vs, combo))
                if all(tuple(subst(a, s2) for a in l[2]) not in self.facts[l[1]] for l in negl):
                    return True
        return False

## 6–7. Herbrand grounding + DPLL, and resolution refutation

In [5]:
# ============================================================
# 6. HERBRAND GROUNDING + DPLL  (complete for function-free KBs)
# ------------------------------------------------------------
# KB ⊨ Q   iff   KB ∧ ¬Q is unsatisfiable.  After Skolemisation, if no
# function symbols remain, the Herbrand universe is the finite set of
# constants, so grounding all clauses over it gives an equisatisfiable
# propositional problem (Herbrand's theorem).  That problem is decided by a
# DPLL search with unit propagation (two watched literals) and
# chronological backtracking – a classical depth-first search over truth
# assignments.  With function symbols the universe is built level by level
# (depth-bounded) – then "unsat" is still a sound proof.
# ============================================================

def dpll_satisfiable(clauses, nvars, stats=None):
    """clauses: list of lists of non-zero ints. Returns True if satisfiable."""
    assign = [0] * (nvars + 1)
    watches = defaultdict(list)
    cls = []
    trail = []
    units = []
    for c in clauses:
        if not c:
            return False
        if len(c) == 1:
            units.append(c[0])
        else:
            idx = len(cls)
            cls.append(list(c))
            watches[c[0]].append(idx)
            watches[c[1]].append(idx)

    def value(l):
        v = assign[abs(l)]
        return v if l > 0 else -v

    def enqueue(l):
        v = value(l)
        if v == -1:
            return False
        if v == 0:
            assign[abs(l)] = 1 if l > 0 else -1
            trail.append(l)
        return True

    for u in units:
        if not enqueue(u):
            return False

    qhead = 0

    def propagate():
        nonlocal qhead
        while qhead < len(trail):
            p = trail[qhead]
            qhead += 1
            false_lit = -p
            wl = watches[false_lit]
            i = 0
            while i < len(wl):
                ci = wl[i]
                c = cls[ci]
                if c[0] == false_lit:
                    c[0], c[1] = c[1], c[0]
                if value(c[0]) == 1:
                    i += 1
                    continue
                found = False
                for k in range(2, len(c)):
                    if value(c[k]) != -1:
                        c[1], c[k] = c[k], c[1]
                        watches[c[1]].append(ci)
                        wl[i] = wl[-1]
                        wl.pop()
                        found = True
                        break
                if found:
                    continue
                if value(c[0]) == -1:
                    return False                   # conflict
                enqueue(c[0])                      # unit
                i += 1
        return True

    # static variable order: most frequent first (VSIDS-like initial activity)
    freq = defaultdict(int)
    for c in clauses:
        for l in c:
            freq[abs(l)] += 1
    order = sorted(range(1, nvars + 1), key=lambda v: -freq[v])
    levels = []          # (trail_len_before, decision_lit, flipped)
    pos = 0
    decisions = 0
    while True:
        if propagate():
            while pos < len(order) and assign[order[pos]] != 0:
                pos += 1
            if pos == len(order):
                if stats is not None:
                    stats['decisions'] = stats.get('decisions', 0) + decisions
                return True
            v = order[pos]
            decisions += 1
            levels.append((len(trail), -v, False, pos))   # try False first
            enqueue(-v)
        else:
            # backtrack chronologically
            while True:
                if not levels:
                    if stats is not None:
                        stats['decisions'] = stats.get('decisions', 0) + decisions
                    return False
                tl, lit, flipped, p0 = levels.pop()
                for l in trail[tl:]:
                    assign[abs(l)] = 0
                del trail[tl:]
                qhead = tl
                pos = p0
                if not flipped:
                    levels.append((tl, -lit, True, p0))
                    enqueue(-lit)
                    break


def _equality_axioms(clauses):
    """reflexivity, symmetry, transitivity, substitutivity (as clauses)."""
    preds = {}
    funcs = {}
    for c in clauses:
        for (_, p, args) in c:
            preds[p] = len(args)
            for a in args:
                _collect_funcs(a, funcs)
    if '=' not in preds:
        return []
    X, Y, Z = '?eqx', '?eqy', '?eqz'
    ax = [frozenset([(True, '=', (X, X))]),
          frozenset([(False, '=', (X, Y)), (True, '=', (Y, X))]),
          frozenset([(False, '=', (X, Y)), (False, '=', (Y, Z)), (True, '=', (X, Z))])]
    for p, n in preds.items():
        if p == '=' or n == 0:
            continue
        for i in range(n):
            xs = ['?eqa%d' % j for j in range(n)]
            ys = list(xs)
            ys[i] = '?eqb'
            ax.append(frozenset([(False, '=', (xs[i], '?eqb')), (False, p, tuple(xs)),
                                 (True, p, tuple(ys))]))
    for f, n in funcs.items():
        for i in range(n):
            xs = ['?eqa%d' % j for j in range(n)]
            ys = list(xs)
            ys[i] = '?eqb'
            ax.append(frozenset([(False, '=', (xs[i], '?eqb')),
                                 (True, '=', ((f,) + tuple(xs), (f,) + tuple(ys)))]))
    return ax


def _collect_funcs(t, funcs):
    if isinstance(t, tuple):
        funcs[t[0]] = len(t) - 1
        for a in t[1:]:
            _collect_funcs(a, funcs)


def _term_depth(t):
    if isinstance(t, str):
        return 0
    return 1 + max((_term_depth(a) for a in t[1:]), default=0)


def herbrand_universe(clauses, depth):
    consts = clause_constants(clauses) or {'$c0'}
    funcs = {}
    for c in clauses:
        for (_, _, args) in c:
            for a in args:
                _collect_funcs(a, funcs)
    U = set(consts)
    for _ in range(depth):
        new = set(U)
        for f, n in funcs.items():
            for combo in itertools.product(sorted(U, key=str), repeat=n):
                new.add((f,) + combo)
        U = new
    return sorted(U, key=lambda t: (_term_depth(t), str(t)))


class GroundingTooLarge(Exception):
    pass


def ground_and_solve(clauses, universe, limit, stats=None):
    """Returns True if the grounded clause set is satisfiable."""
    atom_id = {}
    ground = []
    total = 0
    for c in clauses:
        vs = sorted(clause_vars(c))
        lits = list(c)
        n = len(universe) ** len(vs)
        total += n
        if total > limit:
            raise GroundingTooLarge()
        for combo in (itertools.product(universe, repeat=len(vs)) if vs else [()]):
            s = dict(zip(vs, combo))
            g = set()
            taut = False
            for (sg, p, args) in lits:
                key = (p, tuple(subst(a, s) for a in args) if vs else args)
                i = atom_id.get(key)
                if i is None:
                    i = len(atom_id) + 1
                    atom_id[key] = i
                l = i if sg else -i
                if -l in g:
                    taut = True
                    break
                g.add(l)
            if not taut:
                ground.append(list(g))
    if stats is not None:
        stats['ground_clauses'] = stats.get('ground_clauses', 0) + len(ground)
    return dpll_satisfiable(ground, len(atom_id), stats)


# ============================================================
# 7. RESOLUTION REFUTATION  (given-clause loop, set of support,
#    best-first / uniform-cost selection on clause weight, factoring,
#    tautology + duplicate deletion).  Used when function symbols make
#    the Herbrand universe infinite.
# ============================================================

def _rename(c, tag):
    s = {}
    for v in clause_vars(c):
        s[v] = v + tag
    return frozenset(subst_lit(l, s) for l in c) if s else c


def _weight(c):
    return sum(1 + sum(_term_size(a) for a in l[2]) for l in c)


def _term_size(t):
    if isinstance(t, str):
        return 1
    return 1 + sum(_term_size(a) for a in t[1:])


def _canon(c):
    lits = sorted(c, key=lambda l: (l[0], l[1], _shape(l[2])))
    s, n = {}, 0
    out = []
    for l in lits:
        args = []
        for a in l[2]:
            args.append(_canon_t(a, s))
        out.append((l[0], l[1], tuple(args)))
    return frozenset(out)


def _shape(args):
    return str(tuple(_shape_t(a) for a in args))


def _shape_t(t):
    if isinstance(t, str):
        return '?' if t[:1] == '?' else t
    return (t[0],) + tuple(_shape_t(a) for a in t[1:])


def _canon_t(t, s):
    if isinstance(t, str):
        if t[:1] == '?':
            if t not in s:
                s[t] = '?c%d' % len(s)
            return s[t]
        return t
    return (t[0],) + tuple(_canon_t(a, s) for a in t[1:])


def resolution_prove(kb, negq, max_steps=3000, time_limit=2.0, max_depth=4,
                     max_len=10, want_proof=False):
    """Returns (proved: bool, proof_lines)."""
    t0 = time.perf_counter()
    parents = {}
    seen = set()
    usable = []
    index = defaultdict(list)          # (sign,pred) -> clauses in usable
    heap = []
    cnt = itertools.count()

    def add_sos(c, par):
        k = _canon(c)
        if k in seen:
            return False
        seen.add(k)
        if want_proof:
            parents[k] = par
        heapq.heappush(heap, (_weight(c), next(cnt), k))
        return True

    for c in kb:
        k = _canon(c)
        if k in seen:
            continue
        seen.add(k)
        if want_proof:
            parents[k] = ('kb',)
        usable.append(k)
        for l in k:
            index[(l[0], l[1])].append(k)
    for c in negq:
        add_sos(c, ('goal',))
        if not c:
            return True, []

    steps = 0
    while heap and steps < max_steps:
        if time.perf_counter() - t0 > time_limit:
            break
        _, _, given = heapq.heappop(heap)
        steps += 1
        g = _rename(given, '_g')
        new = []
        # binary resolution against usable (including itself)
        for li in g:
            for other in index.get((not li[0], li[1]), ()):
                o = _rename(other, '_o%d' % steps)
                for lj in o:
                    if lj[0] == li[0] or lj[1] != li[1]:
                        continue
                    s = unify_args(li[2], lj[2], {})
                    if s is None:
                        continue
                    r = frozenset(subst_lit(l, s) for l in (g - {li}) | (o - {lj}))
                    new.append((r, (given, other)))
        # factoring
        gl = list(g)
        for i in range(len(gl)):
            for j in range(i + 1, len(gl)):
                a, b = gl[i], gl[j]
                if a[0] == b[0] and a[1] == b[1]:
                    s = unify_args(a[2], b[2], {})
                    if s is not None:
                        new.append((frozenset(subst_lit(l, s) for l in g), (given,)))
        usable.append(given)
        for l in given:
            index[(l[0], l[1])].append(given)
        for r, par in new:
            if not r:
                if want_proof:
                    parents[frozenset()] = par
                    return True, _resolution_trace(parents)
                return True, []
            if _is_taut(r) or len(r) > max_len:
                continue
            if any(_term_depth(a) > max_depth for l in r for a in l[2]):
                continue
            add_sos(r, par)
    return False, []


def _resolution_trace(parents):
    order, done = [], set()

    def visit(c):
        if c in done:
            return
        done.add(c)
        p = parents.get(c, ('kb',))
        if p[0] not in ('kb', 'goal'):
            for q in p:
                visit(q)
        order.append(c)
    visit(frozenset())
    num = {c: i + 1 for i, c in enumerate(order)}
    lines = []
    for c in order:
        p = parents.get(c, ('kb',))
        if p[0] in ('kb', 'goal'):
            how = 'premise' if p[0] == 'kb' else 'negated query'
        elif len(p) == 1:
            how = 'factor of %d' % num[p[0]]
        else:
            how = 'resolve %d, %d' % (num[p[0]], num[p[1]])
        lines.append('%3d. %-60s [%s]' % (num[c], clause_str(c), how))
    return lines

## 8. The reasoning agent: strategy selection, `predict()` and `explain()`

In [6]:
# ============================================================
# 8. THE REASONING AGENT  (strategy selection)
# ------------------------------------------------------------
#  parse premises + query  ->  clauses for KB and for ¬query
#  (a) closed_world=True            -> stratified NAF forward chaining
#  (b) KB ∪ ¬Q all definite          -> KB∪¬Q satisfiable (least model) -> False
#  (c) KB definite, ¬Q all-negative  -> forward chaining + query match
#  (d) function-free                 -> Herbrand grounding + DPLL (decides)
#  (e) function symbols              -> depth-bounded grounding + DPLL,
#                                       then resolution refutation
#  Output: True iff the query is entailed, False otherwise (binary task).
# ============================================================

class KB:
    """Parsed + clausified knowledge base (cached per context string)."""

    def __init__(self, premises):
        self.formulas, self.errors = [], []
        for p in split_premises(premises):
            try:
                self.formulas.append(parse_formula(p))
            except FOLParseError as e:
                self.errors.append(str(e))
        self.clauses = []
        for f in self.formulas:
            self.clauses.extend(formula_to_clauses(f))
        self.clauses = list(dict.fromkeys(self.clauses))
        self.has_eq = any(l[1] == '=' for c in self.clauses for l in c)
        if self.has_eq:                   # equality axioms are definite clauses
            self.clauses += _equality_axioms(self.clauses)
        self.definite = all(is_definite(c) for c in self.clauses)
        self.funcs = has_functions(self.clauses)
        self._horn = None
        self._cwa = None
        self._consistent = None

    @property
    def horn(self):
        if self._horn is None:
            self._horn = HornModel(self.clauses)
        return self._horn

    @property
    def cwa(self):
        if self._cwa is None:
            rules = []
            for f in self.formulas:
                r = formula_to_nafrules(f)
                if r is not None:
                    rules.extend(r)
            self._cwa = CWAModel(rules)
        return self._cwa


_KB_CACHE = {}


def get_kb(premises):
    key = premises if isinstance(premises, str) else repr(premises)
    kb = _KB_CACHE.get(key)
    if kb is None:
        if len(_KB_CACHE) > 5000:
            _KB_CACHE.clear()
        kb = KB(premises)
        _KB_CACHE[key] = kb
    return kb


def _parse_query(query):
    qs = split_premises(query) if not isinstance(query, tuple) else [query]
    fs = [parse_formula(q) if isinstance(q, str) else q for q in qs]
    if not fs:
        raise FOLParseError('empty query')
    f = fs[0]
    for g in fs[1:]:
        f = ('and', f, g)
    return f


def _query_cwa_literals(f):
    env = {}
    while f[0] == 'ex':
        env[f[1]] = '?' + f[1]
        f = f[2]
    lits = [_ast_literal(g, env) for g in _flatten_and(f)]
    return None if None in lits else lits


def entails(kb, qf, info):
    cfg = CONFIG
    negq = formula_to_clauses(qf, negate=True)

    # (a) closed world
    if cfg['closed_world']:
        lits = _query_cwa_literals(qf)
        if lits is not None:
            info['method'] = 'forward-chaining (CWA, NAF)'
            return kb.cwa.query(lits)

    all_cl = kb.clauses + negq
    funcs = kb.funcs or has_functions(negq)

    q_eq = any(l[1] == '=' for c in negq for l in c)
    if kb.definite and not funcs and not (q_eq and not kb.has_eq):
        # (b) definite program + definite ¬Q is always satisfiable
        if all(is_definite(c) for c in negq):
            info['method'] = 'horn-satisfiable (least model)'
            return False
        # (c) ¬Q = single all-negative clause  -> ∃-conjunctive query
        if len(negq) == 1 and not any(l[0] for l in negq[0]) and \
                not (kb.horn.unsafe and clause_constants(negq) - kb.horn.consts):
            info['method'] = 'forward-chaining'
            atoms = [(l[1], l[2]) for l in negq[0]]
            s = kb.horn.holds(atoms)
            info['fc_steps'] = kb.horn.steps
            if s is not None:
                info['proof_atoms'] = [(p, tuple(subst(a, s) for a in args)) for p, args in atoms]
            return s is not None

    eq = _equality_axioms(all_cl)
    work = all_cl + eq
    stats = {}
    if not funcs and not has_functions(eq):
        info['method'] = 'grounding + DPLL'
        try:
            U = herbrand_universe(work, 0)
            sat = ground_and_solve(work, U, cfg['max_ground_clauses'], stats)
            info.update(stats)
            return not sat
        except GroundingTooLarge:
            info['method'] = 'resolution (grounding too large)'
            proved, _ = resolution_prove(work[:len(kb.clauses)] + eq, negq,
                                         cfg['resolution_max_steps'], cfg['resolution_time_limit'])
            return proved

    # (e) function symbols: iterative deepening on Herbrand universe depth
    for d in range(cfg['herbrand_max_depth'] + 1):
        try:
            U = herbrand_universe(work, d)
            if not ground_and_solve(work, U, cfg['max_ground_clauses'], stats):
                info['method'] = 'grounding(depth=%d) + DPLL' % d
                info.update(stats)
                return True
        except GroundingTooLarge:
            break
    info['method'] = 'resolution'
    proved, _ = resolution_prove(kb.clauses + eq, negq,
                                 cfg['resolution_max_steps'], cfg['resolution_time_limit'])
    return proved


def predict(premises, query, return_info=False):
    """premises: str | list[str] (FOL or simple English); query: str.
    Returns True iff KB ⊨ query."""
    info = {}
    try:
        premises, query = maybe_translate(premises, query)
        kb = get_kb(premises)
        info['parse_errors'] = kb.errors
        qf = _parse_query(query)
        ans = entails(kb, qf, info)
    except Exception as e:                       # never crash the batch
        info['error'] = '%s: %s' % (type(e).__name__, e)
        ans = CONFIG['default_answer']
    return (ans, info) if return_info else ans


def explain(premises, query):
    """Human-readable reasoning trace for the report."""
    premises, query = maybe_translate(premises, query)
    kb = get_kb(premises)
    qf = _parse_query(query)
    out = ['KNOWLEDGE BASE (parsed):']
    out += ['  %s' % fol_to_str(f) for f in kb.formulas]
    if kb.errors:
        out += ['  [parse errors] ' + e for e in kb.errors]
    out.append('QUERY: %s' % fol_to_str(qf))
    out.append('CLAUSES (KB):')
    out += ['  %s' % clause_str(c) for c in kb.clauses]
    negq = formula_to_clauses(qf, negate=True)
    out.append('CLAUSES (¬query):')
    out += ['  %s' % clause_str(c) for c in negq]
    ans, info = predict(premises, query, return_info=True)
    out.append('METHOD: %s' % info.get('method'))
    if 'proof_atoms' in info:
        out.append('FORWARD-CHAINING PROOF:')
        for a in info['proof_atoms']:
            out += ['  ' + line for line in kb.horn.proof(a)]
    elif ans and not CONFIG['closed_world']:
        eq = _equality_axioms(kb.clauses + negq)
        proved, lines = resolution_prove(kb.clauses + eq, negq, 20000, 10.0, want_proof=True)
        if proved:
            out.append('RESOLUTION REFUTATION:')
            out += ['  ' + l for l in lines]
    out.append('ANSWER: %s' % ans)
    return '\n'.join(out)

## 9. (Fallback only) deterministic English → FOL templates
Only used when an input is plain English, as in the public RuleTaker/CounterBench text. Logic inputs bypass this completely.

In [7]:
# ============================================================
# 9. (OPTIONAL) DETERMINISTIC ENGLISH -> FOL TEMPLATES
# ------------------------------------------------------------
# The evaluation supplies logic directly, so this is NOT used for FOL
# inputs.  It is only a fallback so the public RuleTaker/CounterBench text
# (which has no FOL annotation) can be run through the same reasoning
# engine for experiments.  Pure pattern matching – no generative model.
# ============================================================

_FOL_HINT = re.compile(r'[∀∃→∧∨¬⊕↔⇒≠]|->|=>|<->|:-|\b\w+\s*\([^()]*\)')
_VAR_WORDS = {'someone', 'something', 'they', 'it', 'he', 'she', 'people', 'things',
              'everyone', 'everything', 'somebody', 'anyone', 'anything', 'person',
              'thing', 'them'}
_IRREG = {'has': 'have', 'does': 'do', 'is': 'be', 'goes': 'go', 'are': 'be'}


def looks_like_fol(text):
    if isinstance(text, (list, tuple)):
        text = ' '.join(map(str, text))
    return bool(_FOL_HINT.search(str(text)))


def _verb_base(v):
    if v in _IRREG:
        return _IRREG[v]
    for suf in ('ches', 'shes', 'sses', 'xes', 'zes'):
        if v.endswith(suf):
            return v[:-2]
    if v.endswith('ies') and len(v) > 4:
        return v[:-3] + 'y'
    if v.endswith('s') and not v.endswith('ss'):
        return v[:-1]
    return v


def _ent(s):
    s = s.strip().lower()
    s = re.sub(r'^(the|a|an)\s+', '', s)
    if s in _VAR_WORDS:
        return 'x'
    return re.sub(r'\W+', '_', s).strip('_')


def _pred(word):
    w = re.sub(r'\W+', '_', word.strip().lower()).strip('_')
    return w[:1].upper() + w[1:]


def _atom(neg, pred, args):
    a = '%s(%s)' % (pred, ', '.join(args))
    return ('¬' + a) if neg else a


_SUBJ = r'(?P<subj>(?:the |a |an )?[\w]+(?: [\w]+)?)'


def _simple(sentence, prev=None):
    """One clause -> list of literal strings and its subject (or None)."""
    s = sentence.strip().lower().rstrip('.')
    s = re.sub(r'\s+', ' ', s)
    m = re.match(r'^(?P<subj>.+?) (?:is|are) (?P<neg>not )?(?P<rest>.+)$', s)
    if m and len(m.group('subj').split()) <= 3:
        subj = _ent(m.group('subj'))
        rest = m.group('rest')
        neg = bool(m.group('neg'))
        mm = re.match(r'^(?:a|an) (.+)$', rest)
        if mm:
            return [_atom(neg, _pred(mm.group(1)), [subj])], subj
        parts = [p for p in re.split(r',\s*|\s+and\s+', rest) if p]
        out = []
        for p in parts:
            pn = p.startswith('not ')
            p = p[4:] if pn else p
            out.append(_atom(neg or pn, _pred(p), [subj]))
        return out, subj
    m = re.match(r'^(?P<subj>.+?) (?:does|do) not (?P<v>\w+)(?: (?P<obj>.+))?$', s)
    if m:
        subj = _ent(m.group('subj'))
        args = [subj] + ([_ent(m.group('obj'))] if m.group('obj') else [])
        return [_atom(True, _pred(m.group('v')), args)], subj
    m = re.match(r'^(?P<subj>.+?) (?P<neg>cannot|can not|can) (?P<v>\w+)(?: (?P<obj>.+))?$', s)
    if m:
        subj = _ent(m.group('subj'))
        args = [subj] + ([_ent(m.group('obj'))] if m.group('obj') else [])
        return [_atom(m.group('neg') != 'can', _pred('can_' + m.group('v')), args)], subj
    w = s.split()
    if len(w) >= 2:
        if w[0] in ('the', 'a', 'an'):
            k = next((i for i in range(2, len(w)) if w[i].endswith('s')), None)
        else:
            k = 1
        if k is not None and k < len(w):
            subj_txt, v, obj = ' '.join(w[:k]), w[k], ' '.join(w[k + 1:])
            subj = _ent(subj_txt)
            base = v if subj_txt in ('they', 'people', 'things') else _verb_base(v)
            args = [subj] + ([_ent(obj)] if obj else [])
            return [_atom(False, _pred(base), args)], subj
    return None, None


def _conj(text):
    """'X is big and red and they chase the cat' -> literal list."""
    lits, prev = [], None
    for part in re.split(r'\s+and\s+', text.strip()):
        r, subj = _simple(part)
        if r is None and prev is not None:
            r, subj = _simple('%s is %s' % (prev, part))
        if r is None:
            return None
        lits += r
        prev = 'someone' if subj == 'x' else subj
    return lits


def english_to_fol(sentence):
    s = sentence.strip().rstrip('.').strip()
    low = s.lower()
    m = re.match(r'^if (?P<c>.+?),? then (?P<h>.+)$', low)
    if m:
        body, head = _conj(m.group('c')), _conj(m.group('h'))
        if body is None or head is None:
            return None
        f = '(%s) → (%s)' % (' ∧ '.join(body), ' ∧ '.join(head))
        return ('∀x (%s)' % f) if re.search(r'\(x[,)]|, x\)', f) else f
    m = re.match(r'^(?:all )?(?P<adjs>[\w ,]+?) (?:people|things|persons|animals) are (?P<h>.+)$', low)
    if m:
        adjs = [a for a in re.split(r',\s*|\s+and\s+', m.group('adjs')) if a]
        body = [_atom(False, _pred(a), ['x']) for a in adjs]
        head = _conj('someone is ' + m.group('h'))
        if head is None:
            return None
        return '∀x ((%s) → (%s))' % (' ∧ '.join(body), ' ∧ '.join(head))
    m = re.match(r'^all (?P<a>\w+?)s? are (?P<neg>not )?(?P<b>\w+?)s?$', low)
    if m:
        return '∀x (%s → %s)' % (_atom(False, _pred(m.group('a')), ['x']),
                                 _atom(bool(m.group('neg')), _pred(m.group('b')), ['x']))
    lits = _conj(low)
    if lits is None:
        return None
    return ' ∧ '.join(lits)


def maybe_translate(premises, query):
    """FOL in -> unchanged.  Plain English in -> template translation."""
    if looks_like_fol(premises) or not CONFIG['nl_fallback']:
        return premises, query
    sents = split_premises(premises)
    out = []
    for s in sents:
        f = english_to_fol(s)
        if f is not None:
            out.append(f)
    q = english_to_fol(str(query)) if not looks_like_fol(query) else query
    return out, (q if q is not None else str(query))

## Sanity checks and reasoning traces

In [8]:
SANITY = [
    # the example from the assignment sheet
    ("Young(bob)\nKind(bob)\n∀x (Young(x) → Smart(x))\n∀x ((Smart(x) ∧ Kind(x)) → Successful(x))", "Successful(bob)", True),
    ("Young(bob)\nKind(bob)\n∀x (Young(x) → Smart(x))\n∀x ((Smart(x) ∧ Kind(x)) → Successful(x))", "¬Successful(bob)", False),
    ("young(bob). kind(bob). smart(X) :- young(X). successful(X) :- smart(X), kind(X).", "successful(bob)", True),
    ("forall x. (P(x) -> Q(x))\nP(a)", "exists y. Q(y)", True),
    ("∀x (Dog(x) → ¬Cat(x))\nDog(rex)", "¬Cat(rex)", True),
    ("∀x (Dog(x) → ¬Cat(x))\nDog(rex)", "Cat(rex)", False),
    ("P(a) ∨ Q(a)\n¬P(a)", "Q(a)", True),
    ("P(a) ⊕ Q(a)\nP(a)", "¬Q(a)", True),
    ("∀x (Human(x) → ∃y Mother(y, x))\nHuman(ann)", "∃z Mother(z, ann)", True),
    ("a = b\nP(a)", "P(b)", True),
    ("∀x (P(x) → P(f(x)))\nP(a)", "P(f(f(a)))", True),
    ("Rain -> Wet\n~Wet", "~Rain", True),
    ("∀x ∀y (Parent(x,y) → Anc(x,y))\n∀x ∀y ∀z (Anc(x,y) ∧ Anc(y,z) → Anc(x,z))\nParent(a,b)\nParent(b,c)\nParent(c,d)", "Anc(a,d)", True),
    ("∀x (A(x) ↔ B(x))\n¬B(k)", "¬A(k)", True),
    ("¬∃x (Bird(x) ∧ Fly(x) ∧ Penguin(x))\nBird(pingu) ∧ Penguin(pingu)", "¬Fly(pingu)", True),
    ("∀x (A(x) → B(x) ∨ C(x))\nA(m)\n¬B(m)", "C(m)", True),
    ("∀x (A(x) → B(x) ∨ C(x))\nA(m)", "C(m)", False),
    ("∀x (P(x) → Q(x))", "∀x (¬Q(x) → ¬P(x))", True),
]
ok = 0
for prem, q, gold in SANITY:
    ans, info = predict(prem, q, return_info=True)
    ok += ans == gold
    print('%-4s %-28s pred=%-5s gold=%-5s  via %s' % ('OK' if ans == gold else 'FAIL', q, ans, gold, info.get('method')))
print('\n%d / %d sanity checks passed' % (ok, len(SANITY)))

OK   Successful(bob)              pred=True  gold=True   via forward-chaining
OK   ¬Successful(bob)             pred=False gold=False  via horn-satisfiable (least model)
OK   successful(bob)              pred=True  gold=True   via forward-chaining
OK   exists y. Q(y)               pred=True  gold=True   via forward-chaining
OK   ¬Cat(rex)                    pred=True  gold=True   via grounding + DPLL
OK   Cat(rex)                     pred=False gold=False  via grounding + DPLL
OK   Q(a)                         pred=True  gold=True   via grounding + DPLL
OK   ¬Q(a)                        pred=True  gold=True   via grounding + DPLL
OK   ∃z Mother(z, ann)            pred=True  gold=True   via grounding(depth=1) + DPLL
OK   P(b)                         pred=True  gold=True   via forward-chaining
OK   P(f(f(a)))                   pred=True  gold=True   via grounding(depth=1) + DPLL
OK   ~Rain                        pred=True  gold=True   via grounding + DPLL
OK   Anc(a,d)                   

In [9]:
print(explain(SANITY[0][0], SANITY[0][1]))      # forward-chaining proof tree
print('\n' + '-' * 70 + '\n')
print(explain(SANITY[7][0], SANITY[7][1]))      # resolution refutation trace
print('\n' + '-' * 70 + '\n')
print(explain(SANITY[8][0], SANITY[8][1]))      # Skolem function + resolution

KNOWLEDGE BASE (parsed):
  young(bob)
  kind(bob)
  ∀x (young(x) → smart(x))
  ∀x ((smart(x) ∧ kind(x)) → successful(x))
QUERY: successful(bob)
CLAUSES (KB):
  young(bob)
  kind(bob)
  smart(?v1) ∨ ¬young(?v1)
  ¬kind(?v2) ∨ ¬smart(?v2) ∨ successful(?v2)
CLAUSES (¬query):
  ¬successful(bob)
METHOD: forward-chaining
FORWARD-CHAINING PROOF:
  successful(bob)   [by rule: smart(?v2) ∧ kind(?v2) → successful(?v2)]
    smart(bob)   [by rule: young(?v1) → smart(?v1)]
      young(bob)   [fact]
    kind(bob)   [fact]
ANSWER: True

----------------------------------------------------------------------

KNOWLEDGE BASE (parsed):
  (p(a) ⊕ q(a))
  p(a)
QUERY: ¬q(a)
CLAUSES (KB):
  p(a) ∨ q(a)
  ¬p(a) ∨ ¬q(a)
  p(a)
CLAUSES (¬query):
  q(a)
METHOD: grounding + DPLL
RESOLUTION REFUTATION:
    1. q(a)                                                         [negated query]
    2. ¬p(a) ∨ ¬q(a)                                                [premise]
    3. ¬p(a)                                         

## 10. Data loading and evaluation utilities

In [10]:
# ============================================================
# 10. DATA LOADING (any csv / json / jsonl / parquet / HF dataset)
#     + EVALUATION (accuracy + inference time)
# ============================================================
import pandas as pd

# candidate column names, most specific first (FOL columns preferred)
CTX_COLS = ['premises-FOL', 'premises_fol', 'premises_FOL', 'fol_premises', 'context_fol',
            'kb_fol', 'logic', 'context', 'premises', 'theory', 'kb', 'facts', 'given_info',
            'story', 'passage']
Q_COLS = ['conclusion-FOL', 'conclusion_fol', 'conclusion_FOL', 'fol_conclusion',
          'statement_fol', 'query_fol', 'statement', 'query', 'question', 'hypothesis',
          'conclusion', 'text']
LABEL_COLS = ['flag', 'label', 'answer', 'gold', 'truth', 'target', 'labels', 'gold_label']
ID_COLS = ['id', 'ID', 'Id', 'example_id', 'idx', 'qid', 'uid']


def _pick(cols, candidates):
    low = {c.lower(): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        if c.lower() in low:
            return low[c.lower()]
    return None


def to_bool(v):
    if isinstance(v, bool):
        return v
    if v is None or (isinstance(v, float) and v != v):
        return None
    s = str(v).strip().lower()
    if s in ('true', 't', 'entailment', 'entailed', 'yes', 'y', '1', 'proved', 'valid'):
        return True
    if s in ('false', 'f', 'not entailment', 'not_entailment', 'non-entailment', 'no', 'n',
             '0', 'uncertain', 'unknown', 'neutral', 'contradiction', 'invalid'):
        return False
    return None


def frame_to_rows(df, ctx_col=None, q_col=None, label_col=None, id_col=None):
    cols = list(df.columns)
    ctx_col = ctx_col or _pick(cols, CTX_COLS)
    q_col = q_col or _pick(cols, Q_COLS)
    label_col = label_col if label_col is not None else _pick(cols, LABEL_COLS)
    id_col = id_col or _pick(cols, ID_COLS)
    if ctx_col is None or q_col is None:
        raise ValueError('could not find context/query columns in %s' % cols)
    print('columns used -> id: %s | context: %s | statement: %s | label: %s'
          % (id_col, ctx_col, q_col, label_col))
    rows = []
    for i, r in enumerate(df.to_dict('records')):
        rows.append({'id': r[id_col] if id_col else i,
                     'context': r[ctx_col],
                     'statement': r[q_col],
                     'label': to_bool(r[label_col]) if label_col else None})
    return rows


def load_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.csv',):
        return pd.read_csv(path)
    if ext in ('.tsv',):
        return pd.read_csv(path, sep='\t')
    if ext in ('.jsonl',):
        return pd.read_json(path, lines=True)
    if ext in ('.json',):
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)
    if ext in ('.parquet',):
        return pd.read_parquet(path)
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    raise ValueError('unsupported file type: ' + path)


def evaluate(rows, name='', closed_world=False, verbose=True):
    """Runs the reasoner on every row; prints accuracy and inference time."""
    old = CONFIG['closed_world']
    CONFIG['closed_world'] = closed_world
    _KB_CACHE.clear()
    preds, methods, errors = [], defaultdict(int), 0
    t0 = time.perf_counter()
    for r in rows:
        ans, info = predict(r['context'], r['statement'], return_info=True)
        preds.append(ans)
        methods[info.get('method', 'error')] += 1
        errors += 'error' in info
    elapsed = time.perf_counter() - t0
    CONFIG['closed_world'] = old
    labelled = [(p, r['label']) for p, r in zip(preds, rows) if r['label'] is not None]
    acc = 100.0 * sum(p == g for p, g in labelled) / len(labelled) if labelled else None
    if verbose:
        print('=' * 64)
        print('%s   (%d queries, closed_world=%s)' % (name, len(rows), closed_world))
        if acc is not None:
            print('  Accuracy        : %.2f %%  (%d labelled)' % (acc, len(labelled)))
        print('  Inference time  : %.3f s total | %.3f ms / query'
              % (elapsed, 1000 * elapsed / max(1, len(rows))))
        print('  Methods used    : %s' % dict(methods))
        if errors:
            print('  Rows with errors: %d' % errors)
    return {'name': name, 'n': len(rows), 'accuracy': acc, 'time_s': elapsed,
            'preds': preds, 'methods': dict(methods)}

## 11. Experiments on the public datasets (RuleTaker, FOLIO, CounterBench)
Runs only if the datasets can be downloaded; otherwise it is skipped. FOLIO is evaluated on its FOL annotations. RuleTaker and CounterBench have no FOL, so they go through the deterministic templates; RuleTaker is reported under both closed- and open-world semantics.

In [11]:
# ============================================================
# 11. PUBLIC DATASETS  (RuleTaker, FOLIO, CounterBench from Hugging Face)
# ------------------------------------------------------------
# Needs:  pip install datasets   (and `huggingface-cli login` for the gated
# yale-nlp/FOLIO repo).  Every loader fails soft: if a dataset cannot be
# downloaded the experiment is skipped and the rest of the notebook runs.
# Override the repo ids below if your copies live elsewhere.
# ============================================================

HF_SOURCES = {
    'ruletaker':    ['tasksource/ruletaker'],
    'folio':        ['yale-nlp/FOLIO', 'tasksource/folio'],
    'counterbench': ['CounterBench/CounterBench', 'yuefeichen/CounterBench'],
}
MAX_ROWS_PER_DATASET = 5000      # keep experiment time reasonable


def hf_frame(key, split_pref=('validation', 'dev', 'test', 'train')):
    try:
        from datasets import load_dataset
    except ImportError:
        print('`datasets` not installed -> skipping', key)
        return None
    for repo in HF_SOURCES[key]:
        try:
            ds = load_dataset(repo)
        except Exception as e:
            print('  could not load %s: %s' % (repo, str(e)[:120]))
            continue
        split = next((s for s in split_pref if s in ds), list(ds.keys())[0])
        d = ds[split]
        # ClassLabel ints -> strings so to_bool() can read them
        for col, feat in d.features.items():
            if feat.__class__.__name__ == 'ClassLabel':
                d = d.map(lambda ex, c=col, f=feat: {c: f.int2str(ex[c])})
        df = d.to_pandas()
        print('loaded %s [%s]  rows=%d  columns=%s' % (repo, split, len(df), list(df.columns)))
        if len(df) > MAX_ROWS_PER_DATASET:
            df = df.sample(MAX_ROWS_PER_DATASET, random_state=0).reset_index(drop=True)
        return df
    return None


def counterbench_frame(repo='CounterBench/CounterBench'):
    """The CounterBench repo stores several JSON files with different
    columns, so `load_dataset` cannot merge them.  Download the question
    files (data_*.json) one by one and concatenate them."""
    try:
        from huggingface_hub import list_repo_files, hf_hub_download
    except ImportError:
        print('huggingface_hub not installed -> skipping counterbench')
        return None
    try:
        files = [f for f in list_repo_files(repo, repo_type='dataset')
                 if f.endswith(('.json', '.jsonl')) and not os.path.basename(f).startswith('meta')]
    except Exception as e:
        print('  could not list %s: %s' % (repo, str(e)[:120]))
        return None
    frames = []
    for f in files:
        path = hf_hub_download(repo, f, repo_type='dataset')
        try:
            with open(path, encoding='utf-8') as fh:
                obj = json.load(fh)
        except json.JSONDecodeError:
            with open(path, encoding='utf-8') as fh:
                obj = [json.loads(l) for l in fh if l.strip()]
        if isinstance(obj, dict):                      # {"data": [...]} or {id: row}
            lists = [v for v in obj.values() if isinstance(v, list)]
            obj = lists[0] if lists else list(obj.values())
        df = pd.json_normalize(obj)
        df['source_file'] = f
        frames.append(df)
        print('  %s: %d rows, columns=%s' % (f, len(df), list(df.columns)[:12]))
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True)
    print('loaded %s  rows=%d' % (repo, len(df)))
    if len(df) > MAX_ROWS_PER_DATASET:
        df = df.sample(MAX_ROWS_PER_DATASET, random_state=0).reset_index(drop=True)
    return df


# ---- CounterBench: causal story -> twin-network propositional logic -------
# given_info : "We know that A causes not B, B and C together cause D, D causes E.
#               We observed C"
# question   : "Would E occur if not A instead of A?"
# Each variable V gets two atoms: F_V (factual world) and C_V (counterfactual
# world).  Structural equations hold in both worlds:  "A causes not B" gives
# B ↔ ¬A, "B and C together cause D" gives D ↔ (B ∧ C); several separate
# causes of one effect are OR-ed.  Exogenous variables (no causes) are shared
# by the two worlds (C_V ↔ F_V) unless intervened on.  The "instead of" part
# and the observations are facts about the factual world; the antecedent is an
# intervention do(...) that replaces the equation in the counterfactual world.
# Answer "yes" iff the KB entails C_Y (or ¬C_Y for "would Y not occur").
# This is Pearl's abduction-action-prediction, decided by the same engine.

_CB_EDGE = re.compile(
    r'^(?P<c>.+?)\s+(?:together\s+)?(?:directly\s+)?(?:causes?|leads? to|results? in|'
    r'has a direct effect on|have a direct effect on|affects?)\s+(?P<n>not\s+)?(?P<e>[\w-]+)$', re.I)
_CB_Q = re.compile(
    r'would\s+(?P<y>[\w-]+)\s+(?P<qneg>not\s+)?(?:occur|happen)s?\s+if\s+(?P<ante>.+?)\s+'
    r'instead\s+of\s+(?P<fact>.+?)\s*\??$', re.I)


def _cb_literals(text):
    out = []
    for part in re.split(r'\s*(?:,|\band\b)\s*', text.strip().rstrip('.?')):
        part = part.strip()
        if not part:
            continue
        neg = bool(re.match(r'^not\s+', part, re.I))
        v = re.sub(r'^not\s+', '', part, flags=re.I).strip().lower()
        if v:
            out.append((v, not neg))
    return out


def counterbench_to_fol(given, question):
    given = re.sub(r'\s+', ' ', str(given)).strip()
    q = _CB_Q.search(re.sub(r'\s+', ' ', str(question)).strip())
    if q is None:
        return None
    eqs = defaultdict(list)                     # effect -> [(kind, [(var,pos)], negated)]
    nodes, observed = set(), []
    for sent in re.split(r'(?<=[.;])\s+|\.\s*$', given):
        sent = sent.strip().rstrip('.;')
        if not sent:
            continue
        sent = sent.split(':')[-1].strip()
        m = re.match(r'^we (?:observed?|see|saw|know that we observed)\s+(?P<o>.+)$', sent, re.I)
        if m:
            observed += _cb_literals(m.group('o'))
            continue
        sent = re.sub(r'^(?:we know that|it is known that)\s+', '', sent, flags=re.I)
        for clause in re.split(r'\s*,\s*(?:and\s+)?(?:we know that\s+)?', sent):
            m = _CB_EDGE.match(clause.strip())
            if not m:
                continue
            causes = m.group('c')
            kind = 'or' if re.search(r'\bor\b', causes) and not re.search(r'\band\b', causes) else 'and'
            lits = []
            for c in re.split(r'\s+(?:and|or)\s+', causes):
                c = c.strip()
                neg = bool(re.match(r'^not\s+', c, re.I))
                v = re.sub(r'^not\s+', '', c, flags=re.I).strip().lower()
                lits.append((v, not neg))
                nodes.add(v)
            e = m.group('e').lower()
            nodes.add(e)
            eqs[e].append((kind, lits, bool(m.group('n'))))
    if not eqs:
        return None
    ante = dict(_cb_literals(q.group('ante')))
    fact = _cb_literals(q.group('fact'))
    y = q.group('y').lower()
    nodes |= set(ante) | {v for v, _ in fact} | {v for v, _ in observed} | {y}
    A = lambda w, v: '%s_%s' % (w, re.sub(r'\W', '_', v))
    L = lambda w, v, pos: A(w, v) if pos else '¬' + A(w, v)

    def rhs(w, n):
        terms = []
        for kind, lits, eneg in eqs[n]:
            inner = (' ∧ ' if kind == 'and' else ' ∨ ').join(L(w, v, pos) for v, pos in lits)
            terms.append(('¬(%s)' if eneg else '(%s)') % inner)
        return ' ∨ '.join(terms)

    prem = []
    for n in sorted(nodes):
        if n in eqs:
            prem.append('%s ↔ (%s)' % (A('F', n), rhs('F', n)))
        if n in ante:
            prem.append(L('C', n, ante[n]))
        elif n in eqs:
            prem.append('%s ↔ (%s)' % (A('C', n), rhs('C', n)))
        else:
            prem.append('%s ↔ %s' % (A('C', n), A('F', n)))     # shared exogenous
    for v, pos in fact + observed:
        prem.append(L('F', v, pos))
    return prem, L('C', y, not q.group('qneg'))


def rows_counterbench(df):
    cols = list(df.columns)
    print('CounterBench columns:', cols)
    gcol = _pick(cols, ['given_info', 'context', 'premise', 'premises', 'story', 'background',
                        'scenario', 'description'])
    qcol = _pick(cols, ['question', 'query', 'statement'])
    lcol = _pick(cols, ['answer', 'label', 'flag'])
    rows, skipped = [], 0
    for i, r in enumerate(df.to_dict('records')):
        conv = counterbench_to_fol(r.get(gcol, '') if gcol else '', r[qcol])
        if conv is None:
            skipped += 1
            if skipped <= 3:
                print('  [unconverted example]', {k: str(v)[:300] for k, v in r.items()})
            continue
        rows.append({'id': i, 'context': conv[0], 'statement': conv[1],
                     'label': to_bool(r[lcol]) if lcol else None})
    print('CounterBench: converted %d rows, skipped %d (unrecognised format)' % (len(rows), skipped))
    return rows

In [12]:
RUN_PUBLIC_EXPERIMENTS = True
results = []
if RUN_PUBLIC_EXPERIMENTS:
    # ---------- FOLIO (FOL annotations; label True vs {False, Uncertain}) ----------
    df = hf_frame('folio')
    if df is not None:
        rows = frame_to_rows(df)
        results.append(evaluate(rows, 'FOLIO (premises-FOL / conclusion-FOL)'))
    # ---------- RuleTaker ----------
    df = hf_frame('ruletaker')
    if df is not None:
        rows = frame_to_rows(df)
        r_cwa = evaluate(rows, 'RuleTaker (closed world)', closed_world=True)
        r_owa = evaluate(rows, 'RuleTaker (open world)', closed_world=False)
        results += [r_cwa, r_owa]
        dcol = _pick(list(df.columns), ['config', 'depth', 'QDep', 'qdep'])
        if dcol:
            print('\nRuleTaker accuracy by %s (closed world):' % dcol)
            g = defaultdict(lambda: [0, 0])
            for p, r, d in zip(r_cwa['preds'], rows, df[dcol]):
                if r['label'] is not None:
                    g[d][0] += p == r['label']; g[d][1] += 1
            for d in sorted(g, key=str):
                print('  %-22s %6.2f %%  (n=%d)' % (d, 100 * g[d][0] / g[d][1], g[d][1]))
    # ---------- CounterBench ----------
    df = counterbench_frame()
    if df is not None:
        rows = rows_counterbench(df)
        if rows:
            results.append(evaluate(rows, 'CounterBench (twin-network counterfactual logic)'))
if results:
    display(pd.DataFrame([{k: r[k] for k in ('name', 'n', 'accuracy', 'time_s')} for r in results]))

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

  could not load yale-nlp/FOLIO: Dataset 'yale-nlp/FOLIO' is a gated dataset on the Hub. You must be authenticated to access it.


README.md:   0%|          | 0.00/869 [00:00<?, ?B/s]

folio_v2_train.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

folio_v2_validation.jsonl:   0%|          | 0.00/208k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

... (14 outputs left out)

  data_balanced_alpha_V1.json: 1000 rows, columns=['question', 'given_info', 'answer', 'type', 'question_id', 'meta.story_id', 'meta.rung', 'meta.query_type', 'meta.model_id', 'meta.graph_id', 'source_file']


data_balanced_backdoor_V2.json:   0%|          | 0.00/117k [00:00<?, ?B/s]

  data_balanced_backdoor_V2.json: 200 rows, columns=['question', 'given_info', 'answer', 'type', 'question_id', 'meta.story_id', 'meta.rung', 'meta.query_type', 'meta.model_id', 'meta.graph_id', 'source_file']
loaded CounterBench/CounterBench  rows=1200
CounterBench columns: ['question', 'given_info', 'answer', 'type', 'question_id', 'meta.story_id', 'meta.rung', 'meta.query_type', 'meta.model_id', 'meta.graph_id', 'source_file']
  [unconverted example] {'question': 'Would Lumbo occur if not Ziklo and not Blaf?', 'given_info': 'We know that Ziklo causes Blaf and Vork, Blaf causes not Trune, Trune or Vork causes Lumbo.', 'answer': 'yes', 'type': 'joint', 'question_id': '100', 'meta.story_id': 'nonsense_0', 'meta.rung': '3', 'meta.query_type': 'det-counterfactual', 'meta.model_id': '500', 'meta.graph_id': 'graph5', 'source_file': 'data_balanced_alpha_V1.json'}
  [unconverted example] {'question': 'Would Zlim occur if not Glent and not Razz?', 'given_info': 'We know that Glent causes Razz

,name,n,accuracy,time_s
0,FOLIO (premises-FOL / conclusion-FOL),203,84.236453,63.899851
1,RuleTaker (closed world),5000,98.120000,6.097941
2,RuleTaker (open world),5000,75.820000,7.547159
3,CounterBench (twin-network counterfactual logic),700,80.714286,0.649732


## 12. Private test set → `submission.csv`
Set `TEST_PATH` to the file given at evaluation (csv / tsv / json / jsonl / parquet / xlsx). Columns are detected automatically: id (`id`…), knowledge base (`context`, `premises-FOL`, `premises`…), query (`statement`, `conclusion-FOL`, `query`…), and optionally a gold label (`flag`, `label`…). If no test file is found, a small demo file is used so the notebook still runs end to end.

In [13]:
TEST_PATH = 'test.csv'            # <-- path of the private test file
SUBMISSION_PATH = 'submission.csv'
PRED_COLUMN = 'flag'              # column name for the predicted label

if not os.path.exists(TEST_PATH):
    print('TEST_PATH not found -> writing and using demo_test.csv')
    pd.DataFrame([{'id': i + 1, 'context': p, 'statement': q, 'flag': g}
                  for i, (p, q, g) in enumerate(SANITY)]).to_csv('demo_test.csv', index=False)
    TEST_PATH = 'demo_test.csv'

test_df = load_table(TEST_PATH)
test_rows = frame_to_rows(test_df)
res = evaluate(test_rows, 'TEST SET: ' + TEST_PATH, closed_world=CONFIG['closed_world'])

sub = pd.DataFrame({'id': [r['id'] for r in test_rows],
                    PRED_COLUMN: ['True' if p else 'False' for p in res['preds']]})
sub.to_csv(SUBMISSION_PATH, index=False)
print('\nwrote %s  (%d rows)' % (SUBMISSION_PATH, len(sub)))
print('FINAL ACCURACY       :', ('%.2f %%' % res['accuracy']) if res['accuracy'] is not None else 'n/a (no gold labels in test file)')
print('TOTAL INFERENCE TIME : %.4f s  (%.3f ms / query)' % (res['time_s'], 1000 * res['time_s'] / max(1, res['n'])))
sub.head(10)

TEST_PATH not found -> writing and using demo_test.csv
columns used -> id: id | context: context | statement: statement | label: flag
TEST SET: demo_test.csv   (18 queries, closed_world=False)
  Accuracy        : 100.00 %  (18 labelled)
  Inference time  : 0.005 s total | 0.262 ms / query
  Methods used    : {'forward-chaining': 5, 'horn-satisfiable (least model)': 1, 'grounding + DPLL': 10, 'grounding(depth=1) + DPLL': 2}

wrote submission.csv  (18 rows)
FINAL ACCURACY       : 100.00 %
TOTAL INFERENCE TIME : 0.0047 s  (0.262 ms / query)


,id,flag
0,1,True
1,2,False
2,3,True
3,4,True
4,5,True
5,6,False
6,7,True
7,8,True
8,9,True
9,10,True
